# Day 11 / 30 — Persist the hybrid model + `train.py` / `predict.py`

**Project:** 6-class news article categorisation  
**Stack:** joblib · same hybrid pipeline as Day 10 · CLI inference  
**Prerequisite:** `data/news_*_clean.parquet` from Day 8; Day 10 hybrid definition is now in `src/pipeline_news.py`.

---

## What we build today

| Step | What | Why |
|------|------|-----|
| 1 | Fit `build_hybrid_pipeline()` on full train | Same estimator as Day 10, single definition for notebook + scripts |
| 2 | `joblib.dump` a bundle: fitted pipeline + label names | Reload anywhere without retraining |
| 3 | `joblib.load` → verify bit-identical predictions vs in-memory | Catch serialization bugs early |
| 4 | Inference contract: raw `text` → `SpacyPreprocessor` → `text_clean` + `text` | Training used precomputed parquet; **CLI must recreate** `text_clean` the same way |
| 5 | Terminal: `uv run python train.py` · `uv run python predict.py` | Reproducible train + stdin/arg inference |

---

## Build in public (LinkedIn)

Short angles you can post: shipped **joblib** persistence + **two-file CLI** (`train` / `predict`), matched **Day 8 preprocessing** on inference so TF-IDF and spaCy branches see the same inputs as training, and kept the hybrid definition in **one module** so notebooks and scripts never drift.

## 0. Setup & paths

In [1]:
import sys
import tempfile
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score
from sklearn.preprocessing import LabelEncoder

sys.path.insert(0, str(Path('..').resolve()))

from src.inference import load_artifact, predict_one, raw_text_to_frame
from src.pipeline_news import build_hybrid_pipeline

DATA_DIR = Path('..') / 'data'
MODEL_DIR = Path('..') / 'models'
ARTIFACT = MODEL_DIR / 'news_hybrid.joblib'
TRAIN_PQ = DATA_DIR / 'news_train_clean.parquet'
TEST_PQ = DATA_DIR / 'news_test_clean.parquet'

print('Artifact path (same default as train.py):', ARTIFACT)

Artifact path (same default as train.py): ../models/news_hybrid.joblib


## 1. Load data & fit hybrid (in-memory baseline)

In [5]:
!python -m spacy download en_core_web_sm

Using Python 3.13.11 environment at: /Users/NaveenkumarVasudevan/Downloads/Project/NLP/nlp-text-classifier-main/.venv
Resolved 1 package in 1.26s                                          
Prepared 1 package in 690ms                                              
Installed 1 package in 6ms.0 (from https://github.com/explos
 + en-core-web-sm==3.8.0 (from https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl)
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [6]:
if not TRAIN_PQ.is_file():
    raise FileNotFoundError(f'Expected {TRAIN_PQ}. Run day8 parquet export first.')

df_train = pd.read_parquet(TRAIN_PQ)
X_train = df_train[['text_clean', 'text']].copy()
le = LabelEncoder()
y_train = le.fit_transform(df_train['label'])

hybrid = build_hybrid_pipeline()
hybrid.fit(X_train, y_train)
print('Fitted in-memory hybrid ✓')

if TEST_PQ.is_file():
    df_test = pd.read_parquet(TEST_PQ)
    X_test = df_test[['text_clean', 'text']].copy()
    y_test = le.transform(df_test['label'])
    y_pred = hybrid.predict(X_test)
    print(f"Test macro F1: {f1_score(y_test, y_pred, average='macro'):.4f}")
else:
    df_test = X_test = y_test = y_pred = None
    print(f'(No {TEST_PQ} — skip test F1.)')

Fitted in-memory hybrid ✓
Test macro F1: 0.0725


## 2. Save & load with `joblib` — round-trip check

The bundle matches `train.py`: `{"pipeline": ..., "classes": le.classes_}`.

In [7]:
bundle = {'pipeline': hybrid, 'classes': le.classes_}

with tempfile.NamedTemporaryFile(suffix='.joblib', delete=False) as tmp:
    tmp_path = Path(tmp.name)

joblib.dump(bundle, tmp_path)
loaded = joblib.load(tmp_path)
tmp_path.unlink(missing_ok=True)

assert np.array_equal(loaded['classes'], le.classes_)
if y_test is not None:
    p_mem = hybrid.predict(X_test)
    p_disk = loaded['pipeline'].predict(X_test)
    assert np.array_equal(p_mem, p_disk)
    print('Round-trip: predictions match in-memory model ✓')
else:
    sample = X_train.iloc[:5]
    assert np.array_equal(hybrid.predict(sample), loaded['pipeline'].predict(sample))
    print('Round-trip: predictions match on train sample ✓')

Round-trip: predictions match in-memory model ✓


## 3. Inference path used by `predict.py`

`raw_text_to_frame` applies `SpacyPreprocessor` so `text_clean` matches Day 8.

In [8]:
sample_raw = (
    'Scientists at NASA announced a new mission to study exoplanet atmospheres '
    'using the James Webb Space Telescope.'
)
X_inf = raw_text_to_frame(sample_raw)
print('text_clean preview:', X_inf['text_clean'].iloc[0][:120], '...')
label = predict_one(bundle, sample_raw)
print('Predicted:', label)

text_clean preview: scientist nasa announce new mission study exoplanet atmosphere james webb space telescope ...
Predicted: SCIENCE


## 4. Write production artifact (optional)

Uncomment to overwrite `models/news_hybrid.joblib` (same file `train.py` writes).

In [9]:
# MODEL_DIR.mkdir(parents=True, exist_ok=True)
# joblib.dump(bundle, ARTIFACT)
# print('Wrote', ARTIFACT)
# verify = load_artifact(ARTIFACT)
# print('Reload OK, classes:', list(verify['classes']))

print('Skipped writing to disk (uncomment cell above to save).')
print('Or run from repo root:  uv run python train.py')

Skipped writing to disk (uncomment cell above to save).
Or run from repo root:  uv run python train.py


## 5. Day 11 summary

- **One pipeline definition** — `src/pipeline_news.py` — keeps Day 10 and Day 11 aligned.
- **Bundle** — fitted `Pipeline` plus `classes` for human-readable labels after `predict`.
- **Inference** — always rebuild `text_clean` from raw text with `SpacyPreprocessor` (same as Day 8).
- **Next (Day 12):** Streamlit app that calls the same inference helpers.

```bash
# From repository root
uv run python train.py
echo "Apple unveiled a new chip for its laptops" | uv run python predict.py
```

In [10]:
print('=' * 58)
print('  DAY 11 COMPLETE')
print('=' * 58)
print('  Artifact layout: pipeline + classes (joblib)')
print('  Scripts: train.py | predict.py')
print('  Next: Day 12 — Streamlit demo')
print('=' * 58)

  DAY 11 COMPLETE
  Artifact layout: pipeline + classes (joblib)
  Scripts: train.py | predict.py
  Next: Day 12 — Streamlit demo
